# 03 · Error analysis

Compares the errors of TF-IDF (word+char) and RoBERTa (seed 42, no augmentation)
on the clean test conditions, then looks at attack failures, and exports a sample
of misclassified texts for **manual coding** (the structured manual error analysis).

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir()) / "src"))
from report_utils import *

ROOT = project_root()
RES, DATA = ROOT / "results", ROOT / "data" / "processed"
FIG, TAB = output_dirs(ROOT)
_, BEST_LR = load_results(RES)
pd.set_option("display.max_colwidth", 120)

KEYS = {"tfidf": "tfidf_word_char", "roberta": "roberta-base"}

def both_models(condition, aug=0.0):
    """One row per text with both detectors' scores and predictions."""
    out = None
    for short, key in KEYS.items():
        p = load_predictions(RES, DATA, key, condition, aug=aug,
                             lr=BEST_LR if key == "roberta-base" else None)
        p = p.rename(columns={"score": f"score_{short}", "pred": f"pred_{short}"})
        out = p if out is None else out.merge(p[["id", f"score_{short}", f"pred_{short}"]], on="id")
    texts = pd.read_parquet(DATA / f"{condition}.parquet", columns=["id", "text"])
    out = out.merge(texts, on="id", how="left")
    out["condition"] = CONDITION_NAMES[condition]
    for short in KEYS:
        out[f"wrong_{short}"] = out[f"pred_{short}"] != out["label"]
    return out

clean = pd.concat([both_models(c) for c in ["test_seen", "test_unseen", "test_reppen"]],
                  ignore_index=True)
clean["class"] = np.where(clean["label"] == 1, "AI", "Human")
clean["words"] = word_count(clean["text"])
print(len(clean), "test predictions loaded")

12036 test predictions loaded


## 1. Do the two detectors make the same mistakes?
For AI texts an error is a miss (false negative); for human texts it is a false accusation (false positive).

In [2]:
def outcome(r):
    t, b = r["wrong_tfidf"], r["wrong_roberta"]
    return ("both wrong" if t and b else "only TF-IDF wrong" if t
            else "only RoBERTa wrong" if b else "both correct")

clean["outcome"] = clean.apply(outcome, axis=1)
agreement = (pd.crosstab([clean["condition"], clean["class"]], clean["outcome"])
             .reindex(columns=["both correct", "only TF-IDF wrong", "only RoBERTa wrong", "both wrong"],
                      fill_value=0))
save_table(agreement, "error_agreement", TAB, "Error overlap between TF-IDF (word+char) and RoBERTa, seed 42")
agreement

saved tables/error_agreement.csv and .tex


outcome              both correct  only TF-IDF wrong  only RoBERTa wrong  \
condition     class                                                        
Held-out gen. AI             1691                269                  17   
              Human          1844                 90                  60   
Rep. penalty  AI              983                819                  11   
              Human          1844                 90                  60   
Seen gen.     AI             1642                352                   5   
              Human          1844                 90                  60   

outcome              both wrong  
condition     class              
Held-out gen. AI             29  
              Human          12  
Rep. penalty  AI            193  
              Human          12  
Seen gen.     AI              7  
              Human          12

## 2. Error rates by domain and by text length

In [3]:
def rates(df, by):
    g = df.groupby(by)
    return pd.DataFrame({
        "Texts": g.size(),
        "TF-IDF error %": g["wrong_tfidf"].mean() * 100,
        "RoBERTa error %": g["wrong_roberta"].mean() * 100,
    }).round(1)

by_domain = rates(clean, ["domain", "class"]).unstack("class")
by_domain.columns = [f"{m} ({c})" for m, c in by_domain.columns]
by_domain = by_domain[[c for c in by_domain.columns if not c.startswith("Texts")]]
save_table(by_domain, "errors_by_domain", TAB,
           "Error rate (%) by domain: misses on AI texts and false accusations on human texts")
by_domain

saved tables/errors_by_domain.csv and .tex


,TF-IDF error % (AI),TF-IDF error % (Human),RoBERTa error % (AI),RoBERTa error % (Human)
domain,,,,
abstracts,29.6,4.2,0.8,0.0
books,23.6,3.7,4.5,11.2
news,31.0,2.6,6.1,1.5
poetry,27.6,3.0,4.8,0.4
recipes,17.3,0.0,4.3,0.4
reddit,31.5,11.2,4.2,6.7
reviews,22.7,9.2,5.0,2.8
wiki,36.3,8.6,5.5,5.2


In [4]:
clean["length"] = pd.cut(clean["words"], [0, 100, 200, 400, np.inf],
                         labels=["<100", "100-199", "200-399", "400+"], right=False)
by_length = rates(clean, ["length", "class"]).unstack("class")
by_length.columns = [f"{m} ({c})" for m, c in by_length.columns]
save_table(by_length, "errors_by_length", TAB, "Error rate (%) by text length in words")
by_length

saved tables/errors_by_length.csv and .tex


/var/folders/w_/1qh_jn_x3_52npdjvpg8ykx80000gn/T/ipykernel_7212/2136670589.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  g = df.groupby(by)


,Texts (AI),Texts (Human),TF-IDF error % (AI),TF-IDF error % (Human),RoBERTa error % (AI),RoBERTa error % (Human)
length,,,,,,
<100,541,60,16.1,45.0,2.6,15.0
100-199,1783,2112,24.1,9.1,4.0,4.7
200-399,3306,2982,30.3,2.6,4.4,2.4
400+,388,864,38.9,1.0,7.5,4.2


## 3. Attack failures: examples
`repr()` shows invisible characters (zero-width spaces `\u200b`, extra line breaks).

In [5]:
atk = pd.concat([both_models(c) for c in ["test_seen_attacks", "test_heldout_attacks"]],
                ignore_index=True)
atk_rates = atk.groupby(["attack", "label"])[["wrong_tfidf", "wrong_roberta"]].mean().unstack("label") * 100
atk_rates.columns = [f"{'TF-IDF' if m.endswith('tfidf') else 'RoBERTa'} "
                     f"{'missed AI' if l == 1 else 'flagged human'} %" for m, l in atk_rates.columns]
atk_rates.round(1)

,TF-IDF flagged human %,TF-IDF missed AI %,RoBERTa flagged human %,RoBERTa missed AI %
attack,,,,
alternative_spelling,4.8,18.8,1.7,0.7
article_deletion,4.2,19.6,0.9,0.8
homoglyph,74.8,3.5,0.1,25.8
insert_paragraphs,5.1,17.9,97.1,0.1
number,5.1,17.7,3.2,0.6
paraphrase,13.7,26.4,35.6,14.0
perplexity_misspelling,5.0,17.8,1.2,1.1
synonym,4.8,24.2,1.2,1.1
upper_lower,5.1,17.9,0.6,1.3


In [6]:
def show(df, n=2, chars=220):
    for _, r in df.head(n).iterrows():
        print(f"[{r['attack']} | {r['domain']} | label={'AI' if r['label'] else 'human'} "
              f"| TF-IDF={r['score_tfidf']:.3f} RoBERTa={r['score_roberta']:.3f}]")
        print(repr(r["text"][:chars]), "\n")

human = atk[atk["label"] == 0].sample(frac=1, random_state=0)
for attack, model in [("insert_paragraphs", "roberta"), ("zero_width_space", "roberta"),
                      ("paraphrase", "roberta"), ("homoglyph", "tfidf")]:
    print(f"##### Human texts wrongly flagged by {model.upper()} under '{attack}'")
    show(human[(human["attack"] == attack) & human[f"wrong_{model}"]])

##### Human texts wrongly flagged by ROBERTA under 'insert_paragraphs'
[insert_paragraphs | abstracts | label=human | TF-IDF=0.628 RoBERTa=1.000]
'  Nolan and Temple Lang argue that "the ability to express statistical\ncomputations is an essential skill." A key related capacity is the ability to\nconduct and present data analysis in a way that another person can unde' 

[insert_paragraphs | reviews | label=human | TF-IDF=0.004 RoBERTa=1.000]
'So a few weeks ago, my sister and I decided to watch a movie and wanted to see Titanic again, so of course we get to the sad ending where Jack dies, but Rose lived on to have an incredible life, I asked my sister "I wond' 

##### Human texts wrongly flagged by ROBERTA under 'zero_width_space'
[zero_width_space | news | label=human | TF-IDF=0.974 RoBERTa=0.999]
"F\u200ba\u200bn\u200bs\u200b \u200bw\u200bh\u200bo\u200b \u200bb\u200bu\u200by\u200b \u200bt\u200bi\u200bc\u200bk\u200be\u200bt\u200bs\u200b \u200bf\u200bo\u200br\u200b \u200bt\u200bh\u200b

## 4. Sample for manual coding
Exports up to 12 errors from each group to `results/error_sample_for_coding.csv`.
Open it in Excel or Numbers, read each text, and fill the **category** column with one
code from the list below (add a short note if useful). An existing file is never
overwritten, so your coding is safe.

| Code | Meaning |
|---|---|
| `formulaic_human` | Human text that is formal, list-like or template-based (recipes, abstracts, wiki) |
| `fluent_ai` | AI text that reads naturally, with no obvious repetition |
| `repetitive_human` | Human text with repetition or rhyme (e.g. poetry, song-like) |
| `short_text` | Too little text to judge (very short) |
| `noisy_formatting` | Odd formatting, broken text or leftover markup |
| `truncated` | Text cut off or ends mid-sentence |
| `topic_or_domain` | Unusual topic or domain-specific vocabulary |
| `label_doubt` | The label itself looks questionable |
| `other` | None of the above (explain in notes) |

In [7]:
errors = clean[clean["outcome"] != "both correct"].copy()
errors["group"] = errors["outcome"] + " · " + errors["class"]
sample = pd.concat([g.sample(n=min(12, len(g)), random_state=0)
                    for _, g in errors.groupby("group")])
sample = sample[["group", "condition", "domain", "model", "class", "score_tfidf",
                 "score_roberta", "words", "text"]].round(4)
sample["text"] = sample["text"].str.slice(0, 2000)
sample["category"] = ""
sample["notes"] = ""

path = RES / "error_sample_for_coding.csv"
if path.exists():
    print(f"{path.name} already exists - not overwritten (delete it to export a new sample).")
else:
    sample.to_csv(path, index=False)
    print(f"Exported {len(sample)} errors to {path.name}")
print(sample["group"].value_counts().to_string())

Exported 72 errors to error_sample_for_coding.csv
group
both wrong · AI               12
both wrong · Human            12
only RoBERTa wrong · AI       12
only RoBERTa wrong · Human    12
only TF-IDF wrong · AI        12
only TF-IDF wrong · Human     12


## 5. Summary of the manual coding (run after filling the categories)

In [8]:
coded = pd.read_csv(RES / "error_sample_for_coding.csv").fillna({"category": ""})
done = coded[coded["category"].str.strip() != ""]
print(f"{len(done)} of {len(coded)} errors coded")
if len(done):
    summary = pd.crosstab(done["category"], done["group"], margins=True, margins_name="Total")
    save_table(summary, "error_categories", TAB, "Manual error categories by error group")
    display(summary)

0 of 72 errors coded
